# Charging List

## 计算三个城市的 CPO 市场占有率

In [1]:
import pandas as pd
import numpy as np

In [3]:
national_charge_station_file = "data/national_charge_station.csv"

national_charge_station_df = pd.read_csv(national_charge_station_file)
national_charge_station_df.head()

/var/folders/2h/m0b45vn951zbzlgkk5953xd00000gr/T/ipykernel_21056/3961359687.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  national_charge_station_df = pd.read_csv(national_charge_station_file)


,provider_place_id,provider_name,preferred_partner,name,longitude,latitude,street,city,region,postal_code,...,typcn_count,typcndc_count,chademo_count,combotyp1_count,combotyp2_count,cee_count,station_name,district,p_ingestday,provider_id
0,101437000_70313,CIS,NaN,快充 小桔充电,113.426215,23.122091,珠园大厦(珠村东环路),广州市,广东省,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,珠村珠园大厦充电站,天河区,2025-08-30,256
1,101437000_70315,CIS,NaN,快充 小桔充电,120.582432,37.020651,云快充汽车充电站(莱西鑫鑫充电站),青岛市,山东省,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,小桔充电汽车充电站（青岛市兴运发风力发电有限公司）,莱西市,2025-08-30,256
2,101437000_70319,CIS,NaN,快充/慢充 小桔充电,110.429794,19.969553,畅快海口悦城小区超级充电站,海口市,海南省,NaN,...,14.0,36.0,NaN,NaN,NaN,NaN,畅快海口悦城小区超级充电站,美兰区,2025-08-30,256
3,101437000_70321,CIS,NaN,快充 小桔充电,110.192638,20.044446,畅快海口珈宝广场超级充电站,海口市,海南省,NaN,...,NaN,22.0,NaN,NaN,NaN,NaN,畅快海口珈宝广场超级充电站,秀英区,2025-08-30,256
4,101437000_70323,CIS,NaN,快充 小桔充电,110.459472,19.899373,云龙镇223国道今典花园,海口市,海南省,NaN,...,NaN,20.0,NaN,NaN,NaN,NaN,今典花园充电站,琼山区,2025-08-30,256


In [4]:
total_cpo_counts = national_charge_station_df["operator_name"].nunique()
total_cpo_counts

378

In [12]:
import pandas as pd

# 讀取全國充電站數據
# 忽略 DtypeWarning，因為我們目前用不到第18列
df_stations = pd.read_csv('data/national_charge_station.csv', low_memory=False)

# 定義目標城市
target_cities = ['成都市', '重慶市', '貴陽市']

# 篩選出目標城市的充電站數據
df_target_cities = df_stations[df_stations['city'].isin(target_cities)].copy()

# 計算目標城市的總充電站數量
total_stations_in_cities = len(df_target_cities)

# 統計每個CPO在目標城市的充電站數量
cpo_station_counts = df_target_cities['operator_name'].value_counts().reset_index()
cpo_station_counts.columns = ['CPO', 'Station_Count']

# 計算市場佔有率
cpo_station_counts['Market_Share (%)'] = (cpo_station_counts['Station_Count'] / total_stations_in_cities * 100).round(2)

# 顯示結果
print(f"分析城市: {', '.join(target_cities)}")
print(f"已知充電站總數: {total_stations_in_cities}")
print("\n--- CPO 市場佔有率 (基於已知數據庫) ---")
print(cpo_station_counts)

分析城市: 成都市, 重慶市, 貴陽市
已知充電站總數: 3360

--- CPO 市場佔有率 (基於已知數據庫) ---
       CPO  Station_Count  Market_Share (%)
0      特来电           1413             42.05
1      蔚景云            596             17.74
2     小桔充电            372             11.07
3     星星充电            271              8.07
4     依威能源            172              5.12
5      云快充            171              5.09
6     国家电网            123              3.66
7    朗新新能源             48              1.43
8     壳牌充电             31              0.92
9      特瓦特             28              0.83
10     逸安启             22              0.65
11    广汽能源             22              0.65
12    太空充电             11              0.33
13    小华充电             10              0.30
14  一码YiMa             10              0.30
15     达克云              5              0.15
16     愉秒充              5              0.15
17    极氪能源              5              0.15
18     润诚达              4              0.12
19     太能充              4              0.12
20    三峡小充   

In [13]:
cpo_station_counts

,CPO,Station_Count,Market_Share (%)
0,特来电,1413,42.05
1,蔚景云,596,17.74
2,小桔充电,372,11.07
3,星星充电,271,8.07
4,依威能源,172,5.12
5,云快充,171,5.09
6,国家电网,123,3.66
7,朗新新能源,48,1.43
8,壳牌充电,31,0.92
9,特瓦特,28,0.83


In [15]:

# --- 計算全國佈局佔比 ---

# 1. 計算每個CPO在全國的總站數
cpo_national_counts = df_stations['operator_name'].value_counts().reset_index()
cpo_national_counts.columns = ['CPO', 'National_Station_Count']

# 2. 將全國總數合併到我們之前的城市統計表中
df_combined_stats = pd.merge(cpo_station_counts, cpo_national_counts, on='CPO', how='left')

# 3. 計算全國佈局佔比
# 這個指標顯示：某CPO的充電站中，有多少百分比是分佈在這三個城市
df_combined_stats['National_Layout_Share (%)'] = \
    (df_combined_stats['Station_Count'] / df_combined_stats['National_Station_Count'] * 100).round(2)

# 為了方便閱讀，重新命名和排序欄位
df_final_stats = df_combined_stats.rename(columns={
    'Station_Count': 'Stations_in_3_Cities',
    'Market_Share (%)': 'Internal_Market_Share (%)'
})

# 重新排序列的順序
df_final_stats = df_final_stats[[
    'CPO',
    'Stations_in_3_Cities',
    'National_Station_Count',
    'Internal_Market_Share (%)',
    'National_Layout_Share (%)'
]]


print("--- CPO 綜合分析 (成都、重慶、貴陽) ---")
print(df_final_stats)


--- CPO 綜合分析 (成都、重慶、貴陽) ---
       CPO  Stations_in_3_Cities  National_Station_Count  \
0      特来电                  1413                   17043   
1      蔚景云                   596                   13045   
2     小桔充电                   372                   17902   
3     星星充电                   271                   14134   
4     依威能源                   172                    6354   
5      云快充                   171                   15464   
6     国家电网                   123                   33324   
7    朗新新能源                    48                    1896   
8     壳牌充电                    31                     119   
9      特瓦特                    28                      79   
10     逸安启                    22                     320   
11    广汽能源                    22                     528   
12    太空充电                    11                      33   
13    小华充电                    10                      10   
14  一码YiMa                    10                     112   
15     达克云  

In [16]:
df_final_stats

,CPO,Stations_in_3_Cities,National_Station_Count,Internal_Market_Share (%),National_Layout_Share (%)
0,特来电,1413,17043,42.05,8.29
1,蔚景云,596,13045,17.74,4.57
2,小桔充电,372,17902,11.07,2.08
3,星星充电,271,14134,8.07,1.92
4,依威能源,172,6354,5.12,2.71
5,云快充,171,15464,5.09,1.11
6,国家电网,123,33324,3.66,0.37
7,朗新新能源,48,1896,1.43,2.53
8,壳牌充电,31,119,0.92,26.05
9,特瓦特,28,79,0.83,35.44


In [21]:

# --- 新增全國影響力佔比 ---

# 1. 計算全國已知充電站總數
total_national_stations = len(df_stations)

# 2. 計算全國影響力佔比
# 這個指標直接反映了CPO在全國範圍內的市場實力
df_final_stats['National_Impact_Share (%)'] = \
    (df_final_stats['National_Station_Count'] / total_national_stations * 100).round(2)

# 3. 重新排定最終列的順序，以獲得最佳可讀性
df_final_stats = df_final_stats[[
    'CPO',
    'Stations_in_3_Cities',
    'Internal_Market_Share (%)',
    'National_Station_Count',
    'National_Layout_Share (%)',
    'National_Impact_Share (%)'
]]
df_final_stats = df_final_stats.sort_values(by='National_Impact_Share (%)', ascending=False).reset_index(drop=True)
print("--- CPO 最終綜合分析 (新增全國影響力佔比) ---")
print(f"全國已知充電站總數: {total_national_stations}")
print(df_final_stats)


--- CPO 最終綜合分析 (新增全國影響力佔比) ---
全國已知充電站總數: 141860
       CPO  Stations_in_3_Cities  Internal_Market_Share (%)  \
0     国家电网                   123                       3.66   
1     小桔充电                   372                      11.07   
2      特来电                  1413                      42.05   
3      云快充                   171                       5.09   
4     星星充电                   271                       8.07   
5      蔚景云                   596                      17.74   
6     南网电动                     1                       0.03   
7     依威能源                   172                       5.12   
8    朗新新能源                    48                       1.43   
9     安悦充电                     1                       0.03   
10    广汽能源                    22                       0.65   
11     逸安启                    22                       0.65   
12     能银链                     2                       0.06   
13    江苏鲸充                     3                       0.09   
14  一码

In [35]:
df_final_stats

,CPO,Stations_in_3_Cities,Internal_Market_Share (%),National_Station_Count,National_Layout_Share (%),National_Impact_Share (%)
0,国家电网,123,3.66,33324,0.37,23.49
1,小桔充电,372,11.07,17902,2.08,12.62
2,特来电,1413,42.05,17043,8.29,12.01
3,云快充,171,5.09,15464,1.11,10.90
4,星星充电,271,8.07,14134,1.92,9.96
5,蔚景云,596,17.74,13045,4.57,9.20
6,南网电动,1,0.03,12103,0.01,8.53
7,依威能源,172,5.12,6354,2.71,4.48
8,朗新新能源,48,1.43,1896,2.53,1.34
9,安悦充电,1,0.03,771,0.13,0.54


In [22]:
logbook_file = "data/Logbook_2026.xlsx"
logbook_df = pd.read_excel(logbook_file)
logbook_df.head()

,City,Day,Date,Station,Use Case,Status,CPO Name,Manufacturer,MODEL,Voltage(V),...,Has_12A_24A,Start Time,Start SoC(%),End Time,End SoC(%),End Method,End Reason,Test Result,Error Describe,Remark
0,成都,1,2026-03-09,大充新都锦水苑充电站,UC01_CC_DC_AC_Sessions,Normal Test,達克雲,浙江萬馬新能源有限公司,DC,1000.0,...,0.0,2026-03-09 11:24:24,73.0,2026-03-09 11:26:42,78.0,APP,Manual Stop,Pass,NaN,實際電流會大於需求電流
1,成都,1,2026-03-09,大邑县铭奇快速充电站,UC01_CC_DC_AC_Sessions,Normal Test,太能沖,成都智邦科技有限公司,DC,220.0,...,0.0,2026-03-09 13:04:35,69.0,2026-03-09 13:07:31,71.0,LAT,Manual Stop,Pass,NaN,NaN
2,成都,1,2026-03-09,蒲江亿谦充电站3号桩,UC01_CC_DC_AC_Sessions,Normal Test,億謙,广东尚晟新能源科技有限公司,DC,1000.0,...,1.0,2026-03-09 14:35:52,56.0,2026-03-09 14:40:33,59.0,LAT,Manual Stop,Pass,NaN,NaN
3,成都,1,2026-03-09,石羊场汽车充换电站,UC01_CC_DC_AC_Sessions,"Station Inaccessible (Access Control, Construc...",NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,成都,1,2026-03-09,億天宸新能源充電站,UC01_CC_DC_AC_Sessions,Normal Test,億天宸,深圳市永联科技股份有限公司,DC,1000.0,...,1.0,2026-03-09 15:58:33,43.0,2026-03-09 16:04:03,46.0,CID,Manual Stop,Pass,NaN,場站cpo無法使用 只能使用雲快充啟動 二維碼失效


In [27]:

from opencc import OpenCC

# --- 讀取並標準化測試日誌 ---

# 1. 讀取 Excel 測試日誌
try:
    df_test_log = pd.read_excel('data/Logbook_2026.xlsx')
    print("成功讀取 Logbook_2026.xlsx")
except FileNotFoundError:
    print("錯誤：找不到 Logbook_2026.xlsx 檔案。")
    # 如果檔案不存在，則停止執行
    df_test_log = None

if df_test_log is not None:
    # 2. 提取唯一的、非空的測試CPO名單
    tested_cpo_list = df_test_log['CPO Name'].dropna().unique()
    print(f"從測試日誌中提取到 {len(tested_cpo_list)} 個唯一的CPO名稱。")

    # 3. 初始化簡繁轉換器 (t2s: Traditional to Simplified)
    cc = OpenCC('t2s')

    # 4. 標準化兩份CPO名單為簡體
    # 轉換資料庫中的CPO名稱
    db_cpo_simplified = {cc.convert(cpo) for cpo in df_final_stats['CPO'].dropna().unique()}
    
    # 轉換測試日誌中的CPO名稱
    tested_cpo_simplified = {cc.convert(cpo) for cpo in tested_cpo_list}

    # 5. 找出在測試名單中，但不在資料庫名單中的CPO
    unmatched_cpos = tested_cpo_simplified - db_cpo_simplified

    print("\n--- 初步匹配結果 (僅簡繁轉換) ---")
    if unmatched_cpos:
        print(f"發現 {len(unmatched_cpos)} 個可能的新CPO或命名不一致的品牌：")
        for cpo in sorted(list(unmatched_cpos)):
            print(f"- {cpo}")
    else:
        print("恭喜！所有測試的CPO在簡繁轉換後都能在資料庫中找到對應。")

    # 為了後續分析，我們也看看匹配上的
    matched_cpos = tested_cpo_simplified.intersection(db_cpo_simplified)
    print(f"\n成功匹配 {len(matched_cpos)} 個CPO。")



成功讀取 Logbook_2026.xlsx
從測試日誌中提取到 77 個唯一的CPO名稱。

--- 初步匹配結果 (僅簡繁轉換) ---
發現 64 個可能的新CPO或命名不一致的品牌：
- Bmw
- Nio
- Tesla
- 一码
- 万净极速充
- 中国石油 金融中心站
- 云谷向黔
- 云鲲新能源
- 亿天宸
- 亿汇充
- 亿谦
- 仁和云朵
- 倢电充
- 公交新能源
- 华之耀
- 南方电网
- 卫莱电
- 吉智能源
- 四川明星新能源
- 四川能源
- 城头能源
- 壳牌
- 太能冲
- 小可乐
- 小朗新能源
- 小橘充电
- 小飞象
- 小马爱充
- 小鹏
- 岚图
- 岳华新能源
- 帕克公源
- 开麦斯
- 快鳗
- 惠知电
- 户户充
- 新电徒
- 昆仑网电
- 昊柏
- 易安电
- 易安装
- 智道
- 极氪
- 武侯商程
- 比亚迪
- 润小电
- 润成达
- 渝快充
- 滨南时代
- 特斯拉
- 理想
- 紫微星科技
- 聚合快充
- 蓝鲤
- 车库电装
- 迅电
- 逸安起
- 道唱充充
- 重庆能源
- 闪开
- 雷曼
- 领充
- 驴充充
- 驿满充电

成功匹配 13 個CPO。


## Station ID

In [31]:
import os
import json
import pandas as pd

# --- 模擬：通過 station_id 合併 Logbook 與圖片元數據 ---

# 步驟 1: 模擬為 Logbook DataFrame 新增 station_id
# 假設 logbook_df 已經從之前的儲存格讀取
if 'logbook_df' in locals():
    # 創建一個副本以避免修改原始df
    logbook_with_sid = logbook_df.copy()
    # 為第一行模擬賦予 station_id
    logbook_with_sid.loc[0, 'station_id'] = 'CD_bai_yuan_zhi_lian_001'
    print("1. 已為 Logbook (副本) 模擬新增 'station_id' 欄位。")
else:
    print("錯誤：'logbook_df' 不存在，請先執行讀取 Logbook 的儲存格。")
    logbook_with_sid = None

if logbook_with_sid is not None:
    # 步驟 2: 遍歷 images 資料夾，讀取所有 meta.json
    images_base_path = 'images/'
    all_meta_data = []

    if os.path.exists(images_base_path):
        # os.walk 會遍歷所有子目錄
        for root, dirs, files in os.walk(images_base_path):
            if 'meta.json' in files:
                station_id = os.path.basename(root)
                meta_path = os.path.join(root, 'meta.json')
                try:
                    with open(meta_path, 'r', encoding='utf-8') as f:
                        meta_data = json.load(f)
                        meta_data['station_id'] = station_id  # 將資料夾名稱作為 station_id 加入
                        all_meta_data.append(meta_data)
                except (json.JSONDecodeError, UnicodeDecodeError) as e:
                    print(f"警告：讀取 {meta_path} 時出錯: {e}")

    if not all_meta_data:
        print("2. 在 'images/' 資料夾中未找到任何有效的 meta.json 檔案。")
    else:
        print(f"2. 成功讀取 {len(all_meta_data)} 個 meta.json 檔案。")

        # 步驟 3: 創建元數據 DataFrame
        meta_df = pd.DataFrame(all_meta_data)
        print("3. 已根據 meta.json 創建元數據 DataFrame。")

        # 步驟 4: 使用 station_id 進行合併
        # 使用左合併，保留所有 Logbook 記錄
        merged_df = pd.merge(logbook_with_sid, meta_df, on='station_id', how='left')
        print("4. 已成功將 Logbook 數據與元數據合併。")

        # 步驟 5: 顯示合併結果的關鍵欄位
        print("\n--- 合併結果預覽 ---")
        # 為了清晰顯示，我們只選擇部分關鍵欄位
        display_columns = [
            'City', 'Station', 'CPO Name', 'station_id',  # 來自 Logbook
            'cpo_name', 'manufacturer'  # 來自 meta.json
        ]
        # 過濾掉合併後不存在的欄位
        display_columns = [col for col in display_columns if col in merged_df.columns]
        
        if not merged_df.empty:
            print(merged_df[display_columns].head())
        else:
            print("合併後的 DataFrame 為空。")


1. 已為 Logbook (副本) 模擬新增 'station_id' 欄位。
2. 成功讀取 57 個 meta.json 檔案。
3. 已根據 meta.json 創建元數據 DataFrame。
4. 已成功將 Logbook 數據與元數據合併。

--- 合併結果預覽 ---
  City     Station CPO Name                station_id
0   成都  大充新都锦水苑充电站      達克雲  CD_bai_yuan_zhi_lian_001
1   成都  大邑县铭奇快速充电站      太能沖                       NaN
2   成都  蒲江亿谦充电站3号桩       億謙                       NaN
3   成都   石羊场汽车充换电站      NaN                       NaN
4   成都   億天宸新能源充電站      億天宸                       NaN


In [32]:
# Cell 2: 自動生成 station_id
import pandas as pd
from pypinyin import pinyin, Style
from opencc import OpenCC

# 確保 logbook_df 已被讀取
if 'logbook_df' in locals():
    # 創建一個副本以進行操作
    df = logbook_df.copy()

    # --- 步驟 1: 定義城市縮寫和簡繁轉換器 ---
    city_pinyin_map = {
        '成都': 'CD',
        '重庆': 'CQ',  # 使用簡體以匹配轉換後的結果
        '贵阳': 'GY'
    }
    cc = OpenCC('t2s') # Traditional to Simplified

    # --- 步驟 2: 定義 CPO 名稱轉拼音的輔助函式 ---
    def cpo_to_pinyin(name):
        if pd.isna(name) or name.strip() == '':
            return "unknown_cpo"
        # 先轉簡體，再轉拼音
        simplified_name = cc.convert(str(name))
        # 移除所有非中文字元和數字，避免影響拼音轉換
        cleaned_name = ''.join(filter(str.isalnum, simplified_name))
        pinyin_list = pinyin(cleaned_name, style=Style.NORMAL)
        return '_'.join([item[0] for item in pinyin_list])

    # --- 步驟 3: 生成 CPO 拼音和城市縮寫欄位 ---
    # 填充 NaN 值以避免分組時出錯
    df['CPO Name'].fillna('Unknown', inplace=True)
    df['City'].fillna('Unknown', inplace=True)
    
    df['cpo_pinyin'] = df['CPO Name'].apply(cpo_to_pinyin)
    # 先將城市轉為簡體再映射
    df['city_abbr'] = df['City'].apply(lambda x: city_pinyin_map.get(cc.convert(x), 'XX'))

    # --- 步驟 4: 為每個分組創建唯一序號 ---
    # cumcount() 會為每個分組內的項目從 0 開始計數，所以我們 +1
    df['station_num'] = df.groupby(['city_abbr', 'cpo_pinyin']).cumcount() + 1

    # --- 步驟 5: 拼接成最終的 station_id ---
    df['station_id'] = df['city_abbr'] + '_' + df['cpo_pinyin'] + '_' + df['station_num'].astype(str).str.zfill(3)

    # --- 顯示結果 ---
    print("--- 自動生成的 station_id 預覽 ---")
    display_cols = ['City', 'CPO Name', 'station_id']
    print(df[display_cols].head())

    # 將生成的結果賦值給一個新的 DataFrame，方便後續使用
    logbook_with_station_id = df

else:
    print("錯誤：'logbook_df' 不存在，請先執行讀取 Logbook 的儲存格。")


--- 自動生成的 station_id 預覽 ---
  City CPO Name             station_id
0   成都      達克雲       CD_da_ke_yun_001
1   成都      太能沖  CD_tai_neng_chong_001
2   成都       億謙         CD_yi_qian_001
3   成都  Unknown         CD_Unknown_001
4   成都      億天宸    CD_yi_tian_chen_001


/var/folders/2h/m0b45vn951zbzlgkk5953xd00000gr/T/ipykernel_21056/2927187087.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['CPO Name'].fillna('Unknown', inplace=True)
/var/folders/2h/m0b45vn951zbzlgkk5953xd00000gr/T/ipykernel_21056/2927187087.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

In [34]:
logbook_with_station_id.to_csv('logbook_with_station_id.csv', index=False, encoding='utf-8-sig')

In [37]:
# --- 將市場份額數據與日誌數據合併 ---

# 確保兩個關鍵的 DataFrame 都存在
if 'logbook_with_station_id' in locals() and 'df_final_stats' in locals():
    
    # 1. 創建副本以安全操作
    df_log = logbook_with_station_id.copy()
    df_stats = df_final_stats.copy()
    
    # 2. 標準化 CPO 名稱以便合併
    # 我們需要一個標準的鍵。這裡我們使用轉換為簡體的 CPO 名稱。
    cc = OpenCC('t2s')
    
    # 為日誌數據創建簡體 CPO 名稱列
    df_log['CPO_simplified'] = df_log['CPO Name'].apply(lambda x: cc.convert(str(x)) if pd.notna(x) else None)
    
    # 為市場份額數據創建簡體 CPO 名稱列
    df_stats['CPO_simplified'] = df_stats['CPO'].apply(lambda x: cc.convert(str(x)) if pd.notna(x) else None)
    
    # 3. 執行左合併
    # 以日誌數據 (df_log) 為基礎，將市場份額信息 (df_stats) 合併進來
    # 我們只選擇需要的幾列進行合併，避免數據冗餘
    stats_to_merge = df_stats[['CPO_simplified', 'Internal_Market_Share (%)', 'National_Impact_Share (%)']]
    
    df_merged_final = pd.merge(
        df_log,
        stats_to_merge,
        on='CPO_simplified',
        how='left'
    )
    
    # 4. 處理未匹配到的 CPO
    # 對於在 df_stats 中找不到的 CPO，其份額字段會是 NaN。我們可以填充一個特定值。
    # 在 Streamlit 端處理 NaN 會更靈活，所以這裡暫時不動。

    # 5. 刪除臨時的簡體 CPO 列
    df_merged_final.drop(columns=['CPO_simplified'], inplace=True)
    
    # 6. 保存最終的數據文件
    output_filename = 'logbook_app_data.csv'
    df_merged_final.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print(f"--- 數據合併完成 ---")
    print(f"已將包含市場份額的最終數據保存到 '{output_filename}'")
    
    # 預覽合併後的關鍵列
    preview_cols = [
        'CPO Name',
        'station_id',
        'Internal_Market_Share (%)',
        'National_Impact_Share (%)'
    ]
    print("\n合併結果預覽：")
    print(df_merged_final[preview_cols].head())

else:
    print("錯誤：無法執行合併。請確保 'logbook_with_station_id' 和 'df_final_stats' 都已成功生成。")


--- 數據合併完成 ---
已將包含市場份額的最終數據保存到 'logbook_app_data.csv'

合併結果預覽：
  CPO Name             station_id  Internal_Market_Share (%)  \
0      達克雲       CD_da_ke_yun_001                       0.15   
1      太能沖  CD_tai_neng_chong_001                        NaN   
2       億謙         CD_yi_qian_001                        NaN   
3  Unknown         CD_Unknown_001                        NaN   
4      億天宸    CD_yi_tian_chen_001                        NaN   

   National_Impact_Share (%)  
0                       0.08  
1                        NaN  
2                        NaN  
3                        NaN  
4                        NaN  


## 爬取电站数据

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
import time
import os
import sys
from tqdm.notebook import tqdm
from shapely.geometry import Polygon, MultiPolygon

# --- 步驟 1: 設定專案環境與路徑 ---

# 假設此 Notebook 位於 'charging_list_app' 的根目錄
# 我們需要手動指定包含 'core' 和 'District' 的 'road_plan_prod' 專案的路徑
# 請根據您的實際情況修改此路徑
road_plan_prod_root = os.path.abspath('../road_plan_prod') 

if not os.path.exists(road_plan_prod_root):
    print(f"❌ 錯誤: 專案路徑 '{road_plan_prod_root}' 不存在。請檢查路徑設定。")
else:
    # 將 'road_plan_prod' 根目錄添加到系統路徑中
    if road_plan_prod_root not in sys.path:
        sys.path.append(road_plan_prod_root)
        print(f"✅ 已將專案根目錄 '{road_plan_prod_root}' 添加到系統路徑")

    # --- 步驟 2: 匯入自訂模組與設定 ---
    try:
        from core.utils import generate_perfect_hex_grid
        print("✅ 成功匯入 'generate_perfect_hex_grid' 函式。")
    except ImportError:
        print("❌ 錯誤: 無法從 'core.utils' 匯入函式。請確保 'road_plan_prod/core/utils.py' 存在。")
        # 如果無法匯入，則停止執行
        raise

# API Key 和其他設定
api_key = "e04966e153cf5a14405299a75cadafdd" # 請確保這是您有效的高德 API Key
keywords = "充电站"

# --- 步驟 3: 定義爬蟲核心函式 ---

def parse_pois_to_dataframe(pois_list):
		"""將從高德 API 返回的 POI 列表解析為 pandas DataFrame。"""
		if not pois_list:
				return pd.DataFrame()
		extracted_data = []
		fields_to_extract = ['id', 'name', 'address', 'cityname', 'adname', "location"]
		biz_ext_fields = ['rating', 'cost', 'open_time', 'opentime2']
		for poi in pois_list:
				poi_data = {field: poi.get(field) for field in fields_to_extract}
				biz_ext_data = poi.get('biz_ext', {})
				if not isinstance(biz_ext_data, dict): biz_ext_data = {}
				for field in biz_ext_fields:
						value = biz_ext_data.get(field)
						if field == 'cost' and isinstance(value, list):
								value = ", ".join(value) if value else ''
						poi_data[f'biz_{field}'] = value
				extracted_data.append(poi_data)
		return pd.DataFrame(extracted_data)

def crawl_city_stations_simplified(hex_grid_gdf, api_key, keywords, simplification_tolerance=0.001):
		"""遍歷六邊形，爬取數據，並處理翻頁及URL過長問題。"""
		all_stations_df_list = []
		for _, hex_row in tqdm(hex_grid_gdf.iterrows(), total=hex_grid_gdf.shape[0], desc="爬取六边形"):
				hex_id = hex_row['hex_id']
				geometry = hex_row['geometry']
				try:
						simplified_geometry = geometry.simplify(simplification_tolerance, preserve_topology=True)
				except Exception:
						simplified_geometry = geometry
				target_polygon = max(simplified_geometry.geoms, key=lambda p: p.area) if isinstance(simplified_geometry, MultiPolygon) else simplified_geometry
				if not target_polygon: continue
				amap_polygon_str = "|".join([f"{lon},{lat}" for lon, lat in zip(*target_polygon.exterior.coords.xy)])
				page = 1
				while True:
						url = f"https://restapi.amap.com/v3/place/polygon?polygon={amap_polygon_str}&keywords={keywords}&key={api_key}&page={page}"
						try:
								response = requests.get(url, timeout=15)
								if response.status_code == 413: break
								response.raise_for_status()
								data = response.json()
								if data.get('status') == '1' and data.get('pois'):
										page_df = parse_pois_to_dataframe(data['pois'])
										page_df['source_hex_id'] = hex_id
										all_stations_df_list.append(page_df)
										if len(data['pois']) < 20: break
										else: page += 1
								else: break
						except requests.exceptions.RequestException: break
						time.sleep(0.1)
		if not all_stations_df_list: return pd.DataFrame()
		final_df = pd.concat(all_stations_df_list, ignore_index=True)
		final_df.drop_duplicates(subset='id', keep='first', inplace=True)
		return final_df

def run_crawling_for_city(city_name, admin_boundaries_gdf, api_key, project_root, radius_km=5):
		"""為指定城市執行完整的爬取流程。"""
		print(f"\n{'='*20}\n🚀 開始處理城市: {city_name}\n{'='*20}")
		city_boundary_gdf = admin_boundaries_gdf[admin_boundaries_gdf['ct_name'].str.contains(city_name[:-1])]
		if city_boundary_gdf.empty:
				print(f"❌ 錯誤: 在 Shapefile 中未找到城市 '{city_name}'。")
				return
		print(f"✅ 成功獲取 '{city_name}' 的邊界。")
		hex_grid_gdf = generate_perfect_hex_grid(city_boundary_gdf.unary_union.bounds, radius_km)
		clipped_hex_grid_gdf = gpd.clip(hex_grid_gdf, city_boundary_gdf)
		if clipped_hex_grid_gdf.empty:
				print(f"⚠️ 警告: '{city_name}' 裁剪後沒有可用的六邊形網格。")
				return
		print(f"✅ 已為 '{city_name}' 生成 {len(clipped_hex_grid_gdf)} 個六邊形網格。")
		city_stations_df = crawl_city_stations_simplified(clipped_hex_grid_gdf, api_key, keywords)
		if city_stations_df.empty:
				print(f"⚠️ 警告: 未能從 '{city_name}' 爬取到任何數據。")
				return
		# 在當前專案的 'data' 目錄下保存結果
		output_dir = 'data'
		os.makedirs(output_dir, exist_ok=True)
		file_city_name = city_name.replace('市', '')
		final_csv_path = os.path.join(output_dir, f'{file_city_name}_all_stations.csv')
		city_stations_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
		print(f"✅ 爬取完成！共為 '{city_name}' 找到 {len(city_stations_df)} 個不重複的充電站。")
		print(f"   數據已保存到: {final_csv_path}")
		display(city_stations_df.head())

# --- 步驟 4: 執行爬取 ---
try:
		district_path = os.path.join(road_plan_prod_root, 'District/district.shp')
		admin_boundaries_gdf = gpd.read_file(district_path)
		
		# 設定要爬取的城市
		target_city_to_crawl = '贵阳市'
		
		# 執行主函式
		run_crawling_for_city(target_city_to_crawl, admin_boundaries_gdf, api_key, road_plan_prod_root)
		
except Exception as e:
		print(f"❌ 執行過程中發生嚴重錯誤: {e}")
